In [ ]:
pip install optuna openpyxl


In [ ]:
pip install plotly kaleido


In [ ]:
 #Parameter Optimization 1 (아무 생각 없이 normalize 한 결과)

In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import MinMaxScaler

# 1) 경로 설정
input_csv     = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# 2) 데이터 로딩 및 기간 필터링
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# 3) 최적화용 입력 변수(지표) 리스트
indicator_cols = [
    '거래량', '변동 %', 'RSI (14일)',
    '볼린저밴드 상단', '볼린저밴드 하단',
    'MACD', 'MACD 시그널',
    'SMA 5일', 'SMA 10일', 'SMA 20일',
    'SMA 60일', 'SMA 120일', 'SMA 200일',
    '가격 상승률 (2주)', '거래량 상승률 (2주)',
    '가격 상승률 (3개월)', '거래량 상승률 (3개월)',
    '가격 상승률 (6개월)', '거래량 상승률 (6개월)',
    '가격 상승률 (1년)', '거래량 상승률 (1년)'
]

# 4) Min–Max 스케일링 (0~1)
scaler = MinMaxScaler()
# 거래량은 로그 변환 후 스케일링
df['거래량_log'] = np.log1p(df['거래량'])
scale_cols = ['거래량_log'] + [c for c in indicator_cols if c != '거래량']
df_norm = df.copy()
df_norm[[*scale_cols]] = scaler.fit_transform(df[scale_cols])

# 5) 백테스트 함수 정의
def backtest(weights, threshold):
    cash, shares = 10_000.0, 0.0
    for idx, row in df_norm.iterrows():
        # score 계산 (거래량_log_norm 포함, 거래량 원본 컬럼은 제외)
        score = sum(
            w * row[col] 
            for w, col in zip(weights, scale_cols)
        )
        price = df.loc[idx, '종가']

        # 매수: 현금만 있고 score > threshold
        if shares == 0 and score > threshold:
            shares = cash / price
            cash = 0.0
        # 매도: 주식만 있고 score < threshold
        elif shares > 0 and score < threshold:
            cash = shares * price
            shares = 0.0

    # 기간 종료 시 전량 청산
    final_value = cash + shares * df.iloc[-1]['종가']
    roi = (final_value) / 10_000.0 * 100
    return roi

# 6) Optuna 목적 함수
def objective(trial):
    # (scale_cols 개수)만큼 가중치 제안
    weights = [
        trial.suggest_uniform(f"w_{i}", -2.0, 2.0)
        for i in range(len(scale_cols))
    ]
    # threshold 제안: [0, 2*N] 구간
    thr = trial.suggest_uniform("threshold", 0.0, 2.0 * len(scale_cols))
    # ROI 최대화
    return backtest(weights, thr)

# 7) 최적화 실행
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# 8) 모든 트라이얼 파라미터 + ROI → DataFrame → Excel 저장
trials_df = study.trials_dataframe().rename(columns={
    'number': 'trial_number',
    'value':  'ROI'
})
# params_* 컬럼만 뽑아서 재정렬
cols = ['trial_number', 'ROI'] + [c for c in trials_df.columns if c.startswith('params_')]
trials_df[cols].to_excel(
    os.path.join(results_folder, "optimization_trials.xlsx"),
    index=False
)

# 9) 최적 파라미터 & ROI 출력
print("⭐️ Best ROI (%):", round(study.best_value, 2))
print("⭐️ Best Params:")
for k, v in study.best_params.items():
    print(f"   {k}: {v:.4f}")

# 10) 최적 파라미터로 다시 백테스트 (검증)
best_weights = [study.best_params[f"w_{i}"] for i in range(len(scale_cols))]
best_thr     = study.best_params["threshold"]
best_roi     = backtest(best_weights, best_thr)
print(f"▶ Validation ROI with best params: {best_roi:.2f}%")


In [ ]:
 #Parameter Optimization 1-2 
#     각각의 StrategyCondition 서브클래스로 매수·매도 시그널을 판별할 조건을 정의합니다:

# 클래스	매수/매도	설명
# RSIBelow(th)	매수	RSI(14일) < th
# BollingerNearLower(buf)	매수	종가 < Lower BB × (1 + buf) (하단 밴드 근처)
# MACDPositive	매수	MACD > MACD 시그널
# MA5AboveMA10	매수	5일 이동평균 > 10일 이동평균
# MA5BelowMA60	매수	5일 이동평균 < 60일 이동평균
# TwoWeekPriceLow(pct)	매수	2주간 누적 상승률 < pct (%)
# RSISell(th)	매도	RSI(14일) > th


In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import MinMaxScaler

# 1) 경로 설정
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# 2) 데이터 로딩 및 기간 필터링
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# 3) 거래량 로그 정규화
df['거래량_log'] = np.log1p(df['거래량'])
# 2주·3개월 가격/거래량 상승률 raw 컬럼은 그대로 사용

# 4) 전략 조건 클래스 정의
class StrategyCondition:
    def check(self, row): raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th

class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row):
        return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct

class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# 5) 백테스트 함수
def backtest(rsi_buy_th, boll_buffer,
             tw_price_th, tm_price_th,
             tw_vol_th, tm_vol_th,
             rsi_sell_th):
    cash, shares = 10_000.0, 0.0

    # 복합 전략 인스턴스
    buy_conds = [
        RSIBelow(rsi_buy_th),
        BollingerNearLower(boll_buffer),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(tw_price_th),
        ThreeMonthPriceLow(tm_price_th),
        TwoWeekVolumeLow(tw_vol_th),
        ThreeMonthVolumeLow(tm_vol_th),
    ]
    sell_conds = [RSISell(rsi_sell_th)]
    
    for _, row in df.iterrows():
        price = row['종가']

        # 매수
        if shares == 0 and all(cond.check(row) for cond in buy_conds):
            shares = cash / price
            cash = 0.0

        # 매도
        elif shares > 0 and any(cond.check(row) for cond in sell_conds):
            cash = shares * price
            shares = 0.0

    # 최종 청산
    final_val = cash + shares * df.iloc[-1]['종가']
    return (final_val - 10_000.0) / 10_000.0 * 100  # ROI (%)

# 6) Optuna 목적 함수
def objective(trial):
    rsi_buy_th    = trial.suggest_uniform("rsi_buy_th",    0.0, 100.0)
    boll_buffer   = trial.suggest_uniform("boll_buffer",   0.0,   0.1)
    tw_price_th   = trial.suggest_uniform("tw_price_th",   0.0,  20.0)
    tm_price_th   = trial.suggest_uniform("tm_price_th",   0.0,  50.0)
    tw_vol_th     = trial.suggest_uniform("tw_vol_th",     0.0, 100.0)
    tm_vol_th     = trial.suggest_uniform("tm_vol_th",     0.0, 300.0)
    rsi_sell_th   = trial.suggest_uniform("rsi_sell_th",   0.0, 100.0)

    return backtest(
        rsi_buy_th, boll_buffer,
        tw_price_th, tm_price_th,
        tw_vol_th, tm_vol_th,
        rsi_sell_th
    )

# 7) 최적화 실행
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# 8) 결과 저장
trials_df = study.trials_dataframe().rename(columns={
    'number': 'trial_number',
    'value':  'ROI'
})
cols = ['trial_number', 'ROI'] + [c for c in trials_df.columns if c.startswith('params_')]
trials_df[cols].to_excel(
    os.path.join(results_folder, "rule_based_optuna_trials.xlsx"),
    index=False
)

# 9) 최적 파라미터 & 검증
print("⭐️ Best ROI (%):", round(study.best_value, 2))
print("⭐️ Best Params:")
for k, v in study.best_params.items():
    print(f"   {k}: {v:.4f}")

best = study.best_params
valid_roi = backtest(
    best['rsi_buy_th'], best['boll_buffer'],
    best['tw_price_th'], best['tm_price_th'],
    best['tw_vol_th'], best['tm_vol_th'],
    best['rsi_sell_th']
)
print(f"▶ Validation ROI with best params: {valid_roi:.2f}%")



In [ ]:
 #Parameter Optimization 1-3
#     각각의 StrategyCondition 서브클래스로 매수·매도 시그널을 판별할 조건을 정의합니다:

# 클래스	매수/매도	설명
# RSIBelow(th)	매수	RSI(14일) < th
# BollingerNearLower(buf)	매수	종가 < Lower BB × (1 + buf) (하단 밴드 근처)
# MACDPositive	매수	MACD > MACD 시그널
# MA5AboveMA10	매수	5일 이동평균 > 10일 이동평균
# MA5BelowMA60	매수	5일 이동평균 < 60일 이동평균
# TwoWeekPriceLow(pct)	매수	2주간 누적 상승률 < pct (%)
# RSISell(th)	매도	RSI(14일) > th

##   여기에다 매크로 vix, high yield spread 반영 (이건 우선 적용)

buy 100% , sell 100%

In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import MinMaxScaler

# ─── 1) 경로 설정 ───────────────────────────────────────────
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ─── 2) 테슬라 데이터 로딩 및 기간 필터링 ─────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# ─── 3) 매크로 데이터 로딩 & 머지 ───────────────────────────
# VIX.csv
vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
vix_df.columns = ['날짜', 'VIX']
# High Yield Spread adjusted.csv
hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
                     parse_dates=[0], encoding='utf-8-sig')
hys_df.columns = ['날짜', 'HYS']

# 날짜 기준으로 병합 (left join)
df = df.merge(vix_df, on='날짜', how='left')
df = df.merge(hys_df, on='날짜', how='left')

# ─── 4) 지표 전처리 ───────────────────────────────────────────
# 거래량 로그 변환 (정규화 없이 그대로 threshold 신호로 사용)
df['거래량_log'] = np.log1p(df['거래량'])
# 나머지 지표들은 Raw 값 그대로 사용 (변동 % 제외)

# ─── 5) 전략 조건 클래스 ────────────────────────────────────
class StrategyCondition:
    def check(self, row): raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th

class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row):
        return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct

class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# ─── 6) 백테스트 함수 (매크로 시그널 포함) ────────────────
def backtest(rsi_buy_th, boll_buffer,
             tw_price_th, tm_price_th,
             tw_vol_th, tm_vol_th,
             rsi_sell_th):
    cash, shares = 10_000.0, 0.0

    # 전략용 조건 객체 생성
    buy_conds = [
        RSIBelow(rsi_buy_th),
        BollingerNearLower(boll_buffer),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(tw_price_th),
        ThreeMonthPriceLow(tm_price_th),
        TwoWeekVolumeLow(tw_vol_th),
        ThreeMonthVolumeLow(tm_vol_th),
    ]
    sell_conds = [RSISell(rsi_sell_th)]

    for _, row in df.iterrows():
        price = row['종가']

#         # ── 1) 매크로 “강제 매도” 신호 ──
#         # VIX <= 30 또는 HYS >= 7 이면 전량 매도
#         if shares > 0 and ((not pd.isna(row['VIX']) and row['VIX'] <= 30)
#                            or (not pd.isna(row['HYS']) and row['HYS'] >= 7)):
#             cash   = shares * price
#             shares = 0.0
#             continue

        # ── 2) 매크로 “강제 매수” 신호 ──
        # VIX >= 50 이면 전액 매수
        if shares == 0 and (not pd.isna(row['VIX']) and row['VIX'] >= 60):
            shares = cash / price
            cash   = 0.0
            continue

        # ── 3) 룰-베이스 신호 ──
        # (1) 매수: 보유 없고 모든 buy_conds 만족
        if shares == 0 and all(cond.check(row) for cond in buy_conds):
            shares = cash / price
            cash   = 0.0

        # (2) 매도: 보유 있고 any(sell_conds) 만족
        elif shares > 0 and any(cond.check(row) for cond in sell_conds):
            cash   = shares * price
            shares = 0.0

    # 최종 청산
    final_val = cash + shares * df.iloc[-1]['종가']
    return (final_val)-10000 / 10_000.0 * 100  # ROI (%)

# ─── 7) Optuna 최적화 ─────────────────────────────────────
def objective(trial):
    return backtest(
        trial.suggest_uniform("rsi_buy_th",  0, 100),
        trial.suggest_uniform("boll_buffer", 0, 0.1),
        trial.suggest_uniform("tw_price_th", 0, 20),
        trial.suggest_uniform("tm_price_th", 0, 50),
        trial.suggest_uniform("tw_vol_th",   0, 100),
        trial.suggest_uniform("tm_vol_th",   0, 300),
        trial.suggest_uniform("rsi_sell_th", 0, 100),
    )

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# ─── 8) 결과 저장 & 출력 ────────────────────────────────────
trials_df = study.trials_dataframe().rename(columns={'number':'trial','value':'ROI'})
cols = ['trial','ROI'] + [c for c in trials_df.columns if c.startswith('params_')]
trials_df[cols].to_excel(os.path.join(results_folder, "macro_optuna_trials.xlsx"),
                        index=False)

print("⭐️ Best ROI (%):", round(study.best_value, 2))
print("⭐️ Best Params:")
for k, v in study.best_params.items():
    print(f"   {k}: {v:.4f}")

valid_roi = backtest(**study.best_params)
print(f"▶ Validation ROI with best params: {valid_roi:.2f}%")

import optuna.visualization as vis

# ─── 9) 최적화 시각화 ───────────────────────────────────────────
# 9-1) Optimization History (Trial vs ROI)
fig1 = vis.plot_optimization_history(study)
# PNG 로 저장하려면 kaleido(또는 Orca) 설치 필요: pip install kaleido
fig1.write_image(os.path.join(results_folder, "opt_history.png"), format="png")

# 9-2) Parameter Importances
fig2 = vis.plot_param_importances(study)
fig2.write_image(os.path.join(results_folder, "param_importance.png"), format="png")

print("✅ Optimization history and parameter importance plots saved:")
print("   •", os.path.join(results_folder, "opt_history.png"))
print("   •", os.path.join(results_folder, "param_importance.png"))



In [ ]:
#위에다가 몇번의 매도 매수가 있었는지 추가

In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import MinMaxScaler

# ─── 1) 경로 설정 ───────────────────────────────────────────
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ─── 2) 데이터 로딩 및 머지 ─────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
vix_df.columns = ['날짜','VIX']
hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
                     parse_dates=[0], encoding='utf-8-sig')
hys_df.columns = ['날짜','HYS']

df = df.merge(vix_df, on='날짜', how='left').merge(hys_df, on='날짜', how='left')
df['거래량_log'] = np.log1p(df['거래량'])

# ─── 3) 전략 조건 클래스 ────────────────────────────────────
class StrategyCondition:
    def check(self, row): raise NotImplementedError()
class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th
class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row): return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)
class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']
class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']
class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']
class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct
class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct
class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct
class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct
class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# ─── 4) 백테스트 함수 (ROI, buy_count, sell_count 반환) ────────────────
def backtest_count(params):
    cash, shares = 10_000.0, 0.0
    total_invested = 10_000.0
    buy_count = 0
    sell_count = 0

    buy_conds = [
        RSIBelow(params['rsi_buy_th']),
        BollingerNearLower(params['boll_buffer']),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(params['tw_price_th']),
        ThreeMonthPriceLow(params['tm_price_th']),
        TwoWeekVolumeLow(params['tw_vol_th']),
        ThreeMonthVolumeLow(params['tm_vol_th']),
    ]
    sell_conds = [RSISell(params['rsi_sell_th'])]

    for _, row in df.iterrows():
        price, vix = row['종가'], row['VIX']

        # (1) 매크로 강제 매수
        if shares == 0 and not pd.isna(vix) and vix >= 60:
            shares = cash / price
            cash = 0.0
            buy_count += 1
            continue

        # (2) 룰 기반 매수
        if shares == 0 and all(cond.check(row) for cond in buy_conds):
            shares = cash / price
            cash = 0.0
            buy_count += 1

        # (3) 룰 기반 매도
        elif shares > 0 and any(cond.check(row) for cond in sell_conds):
            cash = shares * price
            shares = 0.0
            sell_count += 1

    final_val = cash + shares * df.iloc[-1]['종가']
    roi = (final_val - 10_000.0) / 10_000.0 * 100
    return roi, buy_count, sell_count

# ─── 5) Optuna 최적화 ─────────────────────────────────────
def objective(trial):
    return backtest_count({
        'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
        'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
        'tw_price_th': trial.suggest_float("tw_price_th", 0, 20),
        'tm_price_th': trial.suggest_float("tm_price_th", 0, 50),
        'tw_vol_th':   trial.suggest_float("tw_vol_th",   0, 100),
        'tm_vol_th':   trial.suggest_float("tm_vol_th",   0, 300),
        'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
    })[0]

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# ─── 6) 각 trial별 ROI, buy_count, sell_count 수집 ─────────────
records = []
for trial in study.trials:
    roi, bc, sc = backtest_count(trial.params)
    rec = {'trial': trial.number, 'ROI': roi, **{f'params_{k}': v for k, v in trial.params.items()}}
    rec['buy_count']  = bc
    rec['sell_count'] = sc
    records.append(rec)

results_df = pd.DataFrame(records)

# ─── 7) 결과 저장 & 출력 ────────────────────────────────────
excel_path = os.path.join(results_folder, "macro_optuna_trials_with_counts.xlsx")
results_df.to_excel(excel_path, index=False)

print("⭐️ Best ROI (%):", round(study.best_value, 2))
print("⭐️ Best Params:", study.best_params)
print("✅ Trials with buy/sell counts saved to:", excel_path)


In [ ]:
⭐️ Best ROI (%): 357878.73
⭐️ Best Params:
   rsi_buy_th: 86.4067
   boll_buffer: 0.0415
   tw_price_th: 3.3351
   tm_price_th: 15.1196
   tw_vol_th: 43.4293
   tm_vol_th: 217.0325
   rsi_sell_th: 96.3093
▶ Validation ROI with best params: 357878.73%
    

⭐️ Best ROI (%): 269501.09
⭐️ Best Params:
   rsi_buy_th: 47.8733
   boll_buffer: 0.0150
   tw_price_th: 10.2913
   tm_price_th: 23.1772
   tw_vol_th: 70.1934
   tm_vol_th: 225.2615
   rsi_sell_th: 96.8699
▶ Validation ROI with best params: 269501.09%


VIX < 60 with 100

⭐️ Best ROI (%): 221461.74
⭐️ Best Params:
   rsi_buy_th: 51.1987
   boll_buffer: 0.0611
   tw_price_th: 0.1096
   tm_price_th: 33.3657
   tw_vol_th: 83.8030
   tm_vol_th: 48.7716
   rsi_sell_th: 96.9792
▶ Validation ROI with best params: 221461.74%



VIX 30 < <60 with 1000

⭐️ Best ROI (%): 15123.81
⭐️ Best Params:
   rsi_buy_th: 86.3576
   boll_buffer: 0.0938
   tw_price_th: 4.8323
   tm_price_th: 15.0876
   tw_vol_th: 16.6268
   tm_vol_th: 153.2832
   rsi_sell_th: 39.6654
▶ Validation ROI with best params: 15123.81%


VIX 30 < <60 with 100
⭐️ Best ROI (%): 14669.3
⭐️ Best Params:
   rsi_buy_th: 81.4464
   boll_buffer: 0.0898
   tw_price_th: 4.6479
   tm_price_th: 29.1587
   tw_vol_th: 16.4856
   tm_vol_th: 8.1081
   rsi_sell_th: 23.2581
▶ Validation ROI with best params: 14669.30%
    
VIX 30< < 50
# # How much portion to sell and buy optimization
⭐️ Best ROI (%): 131961.59
⭐️ Best Params:
   rsi_buy_th: 32.9508
   boll_buffer: 0.0844
   tw_price_th: 4.0799
   tm_price_th: 27.3716
   tw_vol_th: 43.3509
   tm_vol_th: 133.7565
   rsi_sell_th: 93.5607
▶ Validation ROI with best params: 131961.59%

⭐️ Best ROI (%): 8189.49
⭐️ Best Params:
   rsi_buy_th: 93.3247
   boll_buffer: 0.0888
   tw_price_th: 4.9123
   tm_price_th: 31.7569
   tw_vol_th: 17.0601
   tm_vol_th: 9.6545
   rsi_sell_th: 47.2580
▶ Validation ROI with best params: 8189.49%

# 매번 buy signal 뜰때마다 10,000달러 더 한다. + sell portion of current position

In [ ]:
#이 코드 써

In [ ]:
import os
import pandas as pd
import numpy as np

# 1) 최적 파라미터 (Optuna 결과)
params = {
    'rsi_buy_th': 86.4067,
    'boll_buffer': 0.0415,
    'tw_price_th': 3.3351,
    'tm_price_th': 15.1196,
    'tw_vol_th': 43.4293,
    'tm_vol_th': 217.0325,
    'rsi_sell_th': 96.3093
}

# 2) 경로 설정 (환경에 맞게 수정)
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_vix      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\VIX.csv"
macro_hys      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\High Yied Spread adjusted.csv"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# 3) 데이터 로드 & 병합
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

vix = pd.read_csv(macro_vix, parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜','VIX']
hys = pd.read_csv(macro_hys, parse_dates=[0], encoding='utf-8-sig')
hys.columns = ['날짜','HYS']

df = df.merge(vix, on='날짜', how='left').merge(hys, on='날짜', how='left')

# 4) 전략 조건 클래스
class StrategyCondition:
    def check(self, row): raise NotImplementedError()
class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th
class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row): return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)
class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']
class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']
class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']
class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct
class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct
class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct
class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct
class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# 5) 백테스트 + 로그 기록
def backtest_with_log(params, extra_on_buy=False):
    initial = 10000.0
    cash, shares = initial, 0.0
    total_invested = initial
    logs = []

    buy_conds = [
        RSIBelow(params['rsi_buy_th']),
        BollingerNearLower(params['boll_buffer']),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(params['tw_price_th']),
        ThreeMonthPriceLow(params['tm_price_th']),
        TwoWeekVolumeLow(params['tw_vol_th']),
        ThreeMonthVolumeLow(params['tm_vol_th']),
    ]
    sell_conds = [RSISell(params['rsi_sell_th'])]

    for _, row in df.iterrows():
        date, price, vix = row['날짜'], row['종가'], row['VIX']

        # 1) 매크로 강제 매수 신호
        if shares == 0 and not pd.isna(vix) and vix >= 60:
            invest = 10000.0 if extra_on_buy else cash
            if invest <= 0: continue
            cash += invest if extra_on_buy else 0.0
            total_invested += invest if extra_on_buy else 0.0
            bought = invest / price
            shares += bought
            cash -= invest
            asset = cash + shares*price
            roi   = (asset - initial) / initial * 100
            logs.append([date, "BUY_MACRO", price, bought, cash, asset, roi])
            continue

        # 2) 룰베이스 매수
        if shares == 0 and all(cond.check(row) for cond in buy_conds):
            invest = 10000.0 if extra_on_buy else cash
            if invest <= 0: continue
            cash += invest if extra_on_buy else 0.0
            total_invested += invest if extra_on_buy else 0.0
            bought = invest / price
            shares += bought
            cash -= invest
            asset = cash + shares*price
            roi   = (asset - initial) / initial * 100
            logs.append([date, "BUY", price, bought, cash, asset, roi])

        # 3) 룰베이스 매도
        elif shares > 0 and any(cond.check(row) for cond in sell_conds):
            sold = shares
            cash += sold * price
            shares = 0.0
            asset = cash
            roi   = (asset - initial) / initial * 100
            logs.append([date, "SELL", price, sold, cash, asset, roi])

    # 4) 최종 청산 기록 (FINAL)
    final_price = df.iloc[-1]['종가']
    date = df.iloc[-1]['날짜']
    if shares > 0:
        cash += shares * final_price
        shares = 0.0
    asset = cash
    roi   = (asset - initial) / initial * 100
    logs.append([date, "FINAL", final_price, 0.0, cash, asset, roi])

    return pd.DataFrame(logs, columns=["날짜","액션","가격","수량","현금","총자산","ROI(%)"])

# 6) 실행 후 CSV 저장
log_one   = backtest_with_log(params, extra_on_buy=False)
log_extra = backtest_with_log(params, extra_on_buy=True)

log_one  .to_csv(os.path.join(results_folder, "log_one_time_10000.csv"),     index=False, encoding='utf-8-sig')
log_extra.to_csv(os.path.join(results_folder, "log_extra_10000_each_buy.csv"),index=False, encoding='utf-8-sig')

print("✅ 파일 저장 완료:")
print("   •", os.path.join(results_folder, "log_one_time_10000.csv"))
print("   •", os.path.join(results_folder, "log_extra_10000_each_buy.csv"))


In [ ]:
import os
import pandas as pd
import numpy as np

# 1) 최적 파라미터
params = {
    'rsi_buy_th': 86.4067,
    'boll_buffer': 0.0415,
    'tw_price_th': 3.3351,
    'tm_price_th': 15.1196,
    'tw_vol_th': 43.4293,
    'tm_vol_th': 217.0325,
    'rsi_sell_th': 96.3093
}

# 2) 경로 설정 (환경에 맞게 수정)
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_vix      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\VIX.csv"
macro_hys      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\High Yied Spread adjusted.csv"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# 3) 데이터 로드 & 병합
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

vix = pd.read_csv(macro_vix, parse_dates=[0], encoding='utf-8-sig'); vix.columns = ['날짜','VIX']
hys = pd.read_csv(macro_hys, parse_dates=[0], encoding='utf-8-sig'); hys.columns = ['날짜','HYS']
df = df.merge(vix, on='날짜', how='left').merge(hys, on='날짜', how='left')

# 4) 신호 함수
def buy_signal(row):
    return (
        row['RSI (14일)']  < params['rsi_buy_th']  and
        row['종가']        < row['볼린저밴드 하단'] * (1 + params['boll_buffer']) and
        row['MACD']       > row['MACD 시그널'] and
        row['SMA 5일']    > row['SMA 10일'] and
        row['SMA 5일']    < row['SMA 60일'] and
        row['가격 상승률 (2주)']  < params['tw_price_th'] and
        row['가격 상승률 (3개월)']< params['tm_price_th'] and
        row['거래량 상승률 (2주)']< params['tw_vol_th'] and
        row['거래량 상승률 (3개월)']<params['tm_vol_th'] and
        not pd.isna(row['VIX']) and row['VIX'] >= 60
    )

def sell_signal(row):
    return row['RSI (14일)'] > params['rsi_sell_th']

# 5) 백테스트 + 로그
def backtest_with_log(extra_on_buy=False):
    initial_capital = 10_000.0
    cash, shares = initial_capital, 0.0
    total_invested = initial_capital
    logs = []

    for _, row in df.iterrows():
        date, price = row['날짜'], row['종가']

        # 매수
        if buy_signal(row):
            # 투자액 = 현재 보유 현금 + (추가 10k if extra_on_buy)
            invest = cash + (10_000.0 if extra_on_buy else 0.0)
            if invest <= 0:
                continue
            # 추가 투자금 반영
            if extra_on_buy:
                total_invested += 10_000.0
            # 주식 매수
            bought = invest / price
            shares += bought
            cash = 0.0
            asset = cash + shares * price
            roi   = (asset - total_invested) / total_invested * 100
            logs.append([date, "BUY", price, bought, cash, asset, roi])
            continue

        # 매도
        if shares > 0 and sell_signal(row):
            sold = shares
            cash += sold * price
            shares = 0.0
            asset = cash
            roi   = (asset - total_invested) / total_invested * 100
            logs.append([date, "SELL", price, sold, cash, asset, roi])

    # 최종 청산
    final_price = df.iloc[-1]['종가']
    final_date  = df.iloc[-1]['날짜']
    if shares > 0:
        cash += shares * final_price
        shares = 0.0
    asset = cash
    roi   = (asset - total_invested) / total_invested * 100
    logs.append([final_date, "FINAL", final_price, 0.0, cash, asset, roi])

    return pd.DataFrame(
        logs,
        columns=["날짜","액션","가격","수량","현금","총자산","ROI(%)"]
    )

# 6) 실행 & 저장
log_one   = backtest_with_log(extra_on_buy=False)
log_extra = backtest_with_log(extra_on_buy=True)

log_one.to_csv(os.path.join(results_folder, "log_one_time_10000.csv"), index=False, encoding='utf-8-sig')
log_extra.to_csv(os.path.join(results_folder, "log_extra_10000_each_buy.csv"), index=False, encoding='utf-8-sig')

print("✅ CSV 저장 완료:")
print("  •", os.path.join(results_folder, "log_one_time_10000.csv"))
print("  •", os.path.join(results_folder, "log_extra_10000_each_buy.csv"))


In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# # 0) — 여기에 최적화된 파라미터를 통째로 붙여넣으세요 —
# param_str = """
#    rsi_buy_th: 86.3576
#    boll_buffer: 0.0938
#    tw_price_th: 4.8323
#    tm_price_th: 15.0876
#    tw_vol_th: 16.6268
#    tm_vol_th: 153.2832
#    rsi_sell_th: 39.6654
# """

# # 1) 파라미터 파싱
# params = {}
# for line in param_str.strip().splitlines():
#     key, val = line.split(':')
#     params[key.strip()] = float(val.strip())

# rsi_buy_th  = params['rsi_buy_th']
# boll_buffer = params['boll_buffer']
# tw_price_th = params['tw_price_th']
# tm_price_th = params['tm_price_th']
# tw_vol_th   = params['tw_vol_th']
# tm_vol_th   = params['tm_vol_th']
# rsi_sell_th = params['rsi_sell_th']

# # 2) 경로 설정
# input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
# os.makedirs(results_folder, exist_ok=True)

# # 3) 데이터 로딩 및 필터링
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# # 4) 매크로 데이터 불러와 병합
# vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# vix_df.columns = ['날짜','VIX']
# hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# hys_df.columns = ['날짜','HYS']
# df = df.merge(vix_df, on='날짜', how='left')\
#        .merge(hys_df, on='날짜', how='left')

# # 5) 거래량 로그 변환 (기술적 룰에 사용)
# df['거래량_log'] = np.log1p(df['거래량'])

# # 6) 백테스트 함수 (Fraction 최적화용, 매수 시마다 10k + sell_portion 투입)
# def backtest(buy_frac, sell_frac):
#     cash = 0.0
#     shares = 0.0
#     total_injected = 0.0

#     for _, row in df.iterrows():
#         price = row['종가']

#         # ── 매크로 “강제 매도”
#         if shares > 0 and (
#             (not pd.isna(row['VIX']) and row['VIX'] >= 50) or
#             (not pd.isna(row['HYS']) and row['HYS'] >= 7)
#         ):
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell
#             continue

#         # ── 매크로 “강제 매수”
#         if shares == 0 and (not pd.isna(row['VIX']) and row['VIX'] <= 30):
#             # inject fresh 10k
#             cash += 10_000.0
#             total_injected += 10_000.0
#             # invest fraction
#             invest = cash * buy_frac
#             shares += invest / price
#             cash -= invest
#             continue

#         # ── 기술적 룰 매수
#         if all([
#             shares >= 0,
#             row['RSI (14일)']           <  rsi_buy_th,
#             row['종가']                 <  row['볼린저밴드 하단'] * (1 + boll_buffer),
#             row['MACD']                 >  row['MACD 시그널'],
#             row['SMA 5일']              >  row['SMA 10일'],
#             row['SMA 5일']              <  row['SMA 60일'],
#             row['가격 상승률 (2주)']     <  tw_price_th,
#             row['가격 상승률 (3개월)']   <  tm_price_th,
#             row['거래량 상승률 (2주)']   <  tw_vol_th,
#             row['거래량 상승률 (3개월)'] <  tm_vol_th
#         ]):
#             # inject fresh 10k + sell portion of existing shares
#             cash += 10_000.0
#             total_injected += 10_000.0
#             # sell portion of current shares to raise extra cash
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell
#             # invest fraction of total cash
#             invest = cash * buy_frac
#             shares += invest / price
#             cash -= invest

#         # ── 기술적 룰 매도
#         elif shares > 0 and row['RSI (14일)'] > rsi_sell_th:
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell

#     # 최종 청산
#     final_value = cash + shares * df.iloc[-1]['종가']
#     # ROI 는 총 투입 금액 대비 수익률
#     roi = (final_value - total_injected) / total_injected * 100
#     return roi

# # 7) Fraction 그리드 탐색 & 결과 수집
# fracs   = [i/10 for i in range(1, 11)]
# records = []
# for buy in fracs:
#     for sell in fracs:
#         roi = backtest(buy, sell)
#         records.append({'buy_frac': buy, 'sell_frac': sell, 'ROI (%)': round(roi, 2)})

# # 8) DataFrame 변환 → CSV 저장
# out_df   = pd.DataFrame(records)
# csv_path = os.path.join(results_folder, "macro_rule_fraction_results.csv")
# out_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
# print("✅ CSV 저장 완료:", csv_path)

# # 9) Heatmap 그리기 → PNG 저장
# heatmap_data = out_df.pivot(index='sell_frac', columns='buy_frac', values='ROI (%)')

# plt.figure(figsize=(8, 6))
# sns.heatmap(
#     heatmap_data,
#     annot=True, fmt=".2f", cmap='viridis',
#     cbar_kws={'label': 'ROI (%)'}
# )
# plt.title('Macro+Rule Strategy: Buy/Sell Fraction별 ROI Heatmap')
# plt.xlabel('Buy Fraction')
# plt.ylabel('Sell Fraction')

# png_path = os.path.join(results_folder, "macro_rule_fraction_heatmap.png")
# plt.savefig(png_path, dpi=300, bbox_inches='tight')
# plt.close()
# print("✅ Heatmap PNG 저장 완료:", png_path)

# # ──────────────────────────────────────────────────────────────
# # This is not financial advice, only data analysis.
# # Please consult a qualified financial professional for personalized guidance.


In [ ]:
# Given the optimized parameter and optimized sell & buy fraction. 
# give me the full simulation result

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ─── 0) — 최적화된 파라미터 붙여넣기 —────────────────────────
param_str = """
   rsi_buy_th: 51.1987
   boll_buffer: 0.0611
   tw_price_th: 0.1096
   tm_price_th: 33.3657
   tw_vol_th: 83.8030
   tm_vol_th: 48.7716
   rsi_sell_th: 96.9792
"""

# 파싱
params = {}
for line in param_str.strip().splitlines():
    k, v = line.split(':')
    params[k.strip()] = float(v.strip())

rsi_buy_th   = params['rsi_buy_th']
boll_buffer  = params['boll_buffer']
tw_price_th  = params['tw_price_th']
tm_price_th  = params['tm_price_th']
tw_vol_th    = params['tw_vol_th']
tm_vol_th    = params['tm_vol_th']
rsi_sell_th  = params['rsi_sell_th']

# ─── 1) 경로 설정 ─────────────────────────────────────────────
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ─── 2) 데이터 로딩 & 필터링 ─────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# ─── 3) 매크로 데이터 병합 ─────────────────────────────────────
vix = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜','VIX']
hys = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
                  parse_dates=[0], encoding='utf-8-sig')
hys.columns = ['날짜','HYS']
df = df.merge(vix, on='날짜', how='left').merge(hys, on='날짜', how='left')

# ─── 4) 거래량 로그 변환 ────────────────────────────────────────
df['거래량_log'] = np.log1p(df['거래량'])

# ─── 5) ROI 계산 함수 ─────────────────────────────────────────
def backtest_roi(buy_frac, sell_frac):
    cash = 0.0
    shares = 0.0
    total_injected = 0.0

    for _, row in df.iterrows():
        price = row['종가']

#         # (매크로 강제 매도: HYS>=7만)
#         if shares > 0 and (not np.isnan(row['HYS']) and row['HYS'] >= 7):
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell
#             continue

        # 매크로 강제 매수: VIX>=60
        if shares == 0 and (not np.isnan(row['VIX']) and row['VIX'] >= 60):
            cash += 10_000.0
            total_injected += 10_000.0
            invest = cash * buy_frac
            shares += invest / price
            cash   -= invest
            continue

        # 기술적 룰 매수
        if shares == 0 and all([
            row['RSI (14일)']           < rsi_buy_th,
            row['종가']                 < row['볼린저밴드 하단'] * (1 + boll_buffer),
            row['MACD']                 > row['MACD 시그널'],
            row['SMA 5일']              > row['SMA 10일'],
            row['SMA 5일']              < row['SMA 60일'],
            row['가격 상승률 (2주)']     < tw_price_th,
            row['가격 상승률 (3개월)']   < tm_price_th,
            row['거래량 상승률 (2주)']   < tw_vol_th,
            row['거래량 상승률 (3개월)'] < tm_vol_th
        ]):
            cash += 10_000.0
            total_injected += 10_000.0
            to_sell = shares * sell_frac
            cash   += to_sell * price
            shares -= to_sell
            invest = cash * buy_frac
            shares += invest / price
            cash   -= invest

        # 기술적 룰 매도
        elif shares > 0 and row['RSI (14일)'] > rsi_sell_th:
            to_sell = shares * sell_frac
            cash   += to_sell * price
            shares -= to_sell

    final_value = cash + shares * df.iloc[-1]['종가']
    return (final_value - total_injected) / total_injected * 100

# ─── 6) Fraction 그리드 탐색 & Heatmap ───────────────────────────
fracs = [i/10 for i in range(1,11)]
records = []
for b in fracs:
    for s in fracs:
        records.append({'buy_frac': b, 'sell_frac': s, 'ROI (%)': round(backtest_roi(b,s), 2)})

res_df = pd.DataFrame(records)
res_df.to_csv(os.path.join(results_folder, "fraction_heatmap.csv"), index=False)

heat = res_df.pivot(index='sell_frac', columns='buy_frac', values='ROI (%)')
plt.figure(figsize=(8,6))
sns.heatmap(heat, annot=True, fmt=".2f", cmap='viridis', cbar_kws={'label':'ROI (%)'})
plt.title('Macro+Rule Fraction Heatmap')
plt.xlabel('Buy Fraction'); plt.ylabel('Sell Fraction')
plt.savefig(os.path.join(results_folder, "fraction_heatmap.png"), dpi=300, bbox_inches='tight')
plt.close()

# ─── 7) 최적 비율 시뮬레이션 & 트레이드 로그 (쿨다운30일) ───────────
best = res_df.loc[res_df['ROI (%)'].idxmax()]
best_b, best_s = best['buy_frac'], best['sell_frac']

trade_log = []
cash = shares = total_injected = 0.0
cooldown = 0
COOLDOWN_DAYS = 30

for _, row in df.iterrows():
    date, price = row['날짜'], row['종가']
    if cooldown > 0:
        cooldown -= 1

    # 1) 매크로 매도 (HYS>=7만)
    if shares > 0 and (not np.isnan(row['HYS']) and row['HYS'] >= 7):
        action, trigger = 'SELL_MACRO_HYS', 'HYS>=7'
        sold = shares * best_s
        cash += sold * price
        shares -= sold

    # 2) 매크로 매수
    elif shares == 0 and cooldown == 0 and (not np.isnan(row['VIX']) and row['VIX'] >= 60):
        action, trigger = 'BUY_MACRO_VIX', 'VIX>=60'
        cash += 10_000.0
        total_injected += 10_000.0
        bought = (cash * best_b) / price
        cash -= bought * price
        shares += bought
        cooldown = COOLDOWN_DAYS

    # 3) 기술적 룰 매수
    elif shares == 0 and cooldown == 0 and all([
        row['RSI (14일)']           < rsi_buy_th,
        row['종가']                 < row['볼린저밴드 하단'] * (1 + boll_buffer),
        row['MACD']                 > row['MACD 시그널'],
        row['SMA 5일']              > row['SMA 10일'],
        row['SMA 5일']              < row['SMA 60일'],
        row['가격 상승률 (2주)']     < tw_price_th,
        row['가격 상승률 (3개월)']   < tm_price_th,
        row['거래량 상승률 (2주)']   < tw_vol_th,
        row['거래량 상승률 (3개월)'] < tm_vol_th
    ]):
        action, trigger = 'BUY_RULE', 'TECH_RULE'
        cash += 10_000.0
        total_injected += 10_000.0
        bridge = shares * best_s
        cash += bridge * price
        shares -= bridge
        bought = (cash * best_b) / price
        cash -= bought * price
        shares += bought
        cooldown = COOLDOWN_DAYS

    # 4) 기술적 룰 매도
    elif shares > 0 and row['RSI (14일)'] > rsi_sell_th:
        action, trigger = 'SELL_RULE', 'RSI>sell_th'
        sold = shares * best_s
        cash += sold * price
        shares -= sold

    else:
        continue

    # 기록
    asset = cash + shares * price
    roi_now = (asset - total_injected) / total_injected * 100 if total_injected > 0 else np.nan
    trade_log.append({
        'date':           date,
        'action':         action,
        'trigger':        trigger,
        'price':          round(price,2),
        'shares':         round(bought if 'BUY' in action else sold,6),
        'cash_after':     round(cash,2),
        'cash_used':      round((bought if 'BUY' in action else sold)*price,2),
        'extra_cash':     round(10_000.0 if 'BUY' in action else 0.0,2),
        'cooldown':       cooldown,
        'total_injected': round(total_injected,2),
        'asset_value':    round(asset,2),
        'ROI_so_far(%)':  round(roi_now,2)
    })

# 최종 청산
if shares > 0:
    date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
    action, trigger = 'LIQUIDATE', 'END'
    sold = shares
    cash += sold * price
    shares = 0.0
    asset = cash
    roi_now = (asset - total_injected) / total_injected * 100

    trade_log.append({
        'date':           date,
        'action':         action,
        'trigger':        trigger,
        'price':          round(price,2),
        'shares':         round(sold,6),
        'cash_after':     round(cash,2),
        'cash_used':      round(sold*price,2),
        'extra_cash':     0.0,
        'cooldown':       cooldown,
        'total_injected': round(total_injected,2),
        'asset_value':    round(asset,2),
        'ROI_so_far(%)':  round(roi_now,2)
    })

# ─── 8) 저장 ───────────────────────────────────────────────────
log_df = pd.DataFrame(trade_log)
summary = pd.DataFrame([{
    'buy_frac':       best_b,
    'sell_frac':      best_s,
    'total_injected': round(total_injected,2),
    'final_asset':    round(cash,2),
    'final_ROI_%':    round((cash - total_injected)/total_injected*100,2)
}])

log_df.to_excel(os.path.join(results_folder,"best_trade_log.xlsx"),index=False)
summary.to_excel(os.path.join(results_folder,"best_trade_summary.xlsx"),index=False)
print("✅ Best fraction:", best_b, best_s)
print("✅ Logs & summary saved to Results")


In [ ]:
### ojbect oreineted


In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import MinMaxScaler

# ─── 1) 경로 설정 ───────────────────────────────────────────
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ─── 2) 테슬라 데이터 로딩 및 기간 필터링 ─────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# ─── 3) 매크로 데이터 로딩 & 머지 ───────────────────────────
# VIX.csv
vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
vix_df.columns = ['날짜', 'VIX']
# High Yield Spread adjusted.csv
hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
                     parse_dates=[0], encoding='utf-8-sig')
hys_df.columns = ['날짜', 'HYS']

# 날짜 기준으로 병합 (left join)
df = df.merge(vix_df, on='날짜', how='left')
df = df.merge(hys_df, on='날짜', how='left')

# ─── 4) 지표 전처리 ───────────────────────────────────────────
# 거래량 로그 변환 (정규화 없이 그대로 threshold 신호로 사용)
df['거래량_log'] = np.log1p(df['거래량'])
# 나머지 지표들은 Raw 값 그대로 사용 (변동 % 제외)

# ─── 5) 전략 조건 클래스 ────────────────────────────────────
class StrategyCondition:
    def check(self, row): raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th

class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row):
        return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct

class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# ─── 6) 백테스트 함수 (매크로 시그널 포함) ────────────────
def backtest(rsi_buy_th, boll_buffer,
             tw_price_th, tm_price_th,
             tw_vol_th, tm_vol_th,
             rsi_sell_th):
    cash, shares = 10_000.0, 0.0

    # 전략용 조건 객체 생성
    buy_conds = [
        RSIBelow(rsi_buy_th),
        BollingerNearLower(boll_buffer),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(tw_price_th),
        ThreeMonthPriceLow(tm_price_th),
        TwoWeekVolumeLow(tw_vol_th),
        ThreeMonthVolumeLow(tm_vol_th),
    ]
    sell_conds = [RSISell(rsi_sell_th)]

    for _, row in df.iterrows():
        price = row['종가']

#         # ── 1) 매크로 “강제 매도” 신호 ──
#         # VIX <= 30 또는 HYS >= 7 이면 전량 매도
#         if shares > 0 and ((not pd.isna(row['VIX']) and row['VIX'] <= 30)
#                            or (not pd.isna(row['HYS']) and row['HYS'] >= 7)):
#             cash   = shares * price
#             shares = 0.0
#             continue

        # ── 2) 매크로 “강제 매수” 신호 ──
        # VIX >= 50 이면 전액 매수
        if shares == 0 and (not pd.isna(row['VIX']) and row['VIX'] >= 60):
            shares = cash / price
            cash   = 0.0
            continue

        # ── 3) 룰-베이스 신호 ──
        # (1) 매수: 보유 없고 모든 buy_conds 만족
        if shares == 0 and all(cond.check(row) for cond in buy_conds):
            shares = cash / price
            cash   = 0.0

        # (2) 매도: 보유 있고 any(sell_conds) 만족
        elif shares > 0 and any(cond.check(row) for cond in sell_conds):
            cash   = shares * price
            shares = 0.0

    # 최종 청산
    final_val = cash + shares * df.iloc[-1]['종가']
    return (final_val)-10000 / 10_000.0 * 100  # ROI (%)

# ─── 7) Optuna 최적화 ─────────────────────────────────────
def objective(trial):
    return backtest(
        trial.suggest_uniform("rsi_buy_th",  0, 100),
        trial.suggest_uniform("boll_buffer", 0, 0.1),
        trial.suggest_uniform("tw_price_th", 0, 20),
        trial.suggest_uniform("tm_price_th", 0, 50),
        trial.suggest_uniform("tw_vol_th",   0, 100),
        trial.suggest_uniform("tm_vol_th",   0, 300),
        trial.suggest_uniform("rsi_sell_th", 0, 100),
    )

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

# ─── 8) 결과 저장 & 출력 ────────────────────────────────────
trials_df = study.trials_dataframe().rename(columns={'number':'trial','value':'ROI'})
cols = ['trial','ROI'] + [c for c in trials_df.columns if c.startswith('params_')]
trials_df[cols].to_excel(os.path.join(results_folder, "macro_optuna_trials.xlsx"),
                        index=False)

print("⭐️ Best ROI (%):", round(study.best_value, 2))
print("⭐️ Best Params:")
for k, v in study.best_params.items():
    print(f"   {k}: {v:.4f}")

valid_roi = backtest(**study.best_params)
print(f"▶ Validation ROI with best params: {valid_roi:.2f}%")

import optuna.visualization as vis

# ─── 9) 최적화 시각화 ───────────────────────────────────────────
# 9-1) Optimization History (Trial vs ROI)
fig1 = vis.plot_optimization_history(study)
# PNG 로 저장하려면 kaleido(또는 Orca) 설치 필요: pip install kaleido
fig1.write_image(os.path.join(results_folder, "opt_history.png"), format="png")

# 9-2) Parameter Importances
fig2 = vis.plot_param_importances(study)
fig2.write_image(os.path.join(results_folder, "param_importance.png"), format="png")

print("✅ Optimization history and parameter importance plots saved:")
print("   •", os.path.join(results_folder, "opt_history.png"))
print("   •", os.path.join(results_folder, "param_importance.png"))



In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.preprocessing import MinMaxScaler
import optuna.visualization as vis

# ────────────────────────────────────────────────────────────────
# 0) 경로 & 파라미터 정의
# ────────────────────────────────────────────────────────────────
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ────────────────────────────────────────────────────────────────
# 0) 최적화된 파라미터 붙여넣기
# ────────────────────────────────────────────────────────────────
param_str = """
rsi_buy_th:   51.1987
boll_buffer:  0.0611
tw_price_th:  0.1096
tm_price_th: 33.3657
tw_vol_th:   83.8030
tm_vol_th:   48.7716
rsi_sell_th: 96.9792
"""
params = {}
for line in param_str.strip().splitlines():
    k, v = line.split(':')
    params[k.strip()] = float(v.strip())

# ────────────────────────────────────────────────────────────────
# 1) DataLoader: 가격 + 매크로 데이터 로딩
# ────────────────────────────────────────────────────────────────
class DataLoader:
    def __init__(self, price_csv, vix_csv, hys_csv):
        self.price_csv = price_csv
        self.vix_csv   = vix_csv
        self.hys_csv   = hys_csv

    def load(self):
        df = pd.read_csv(self.price_csv, encoding='utf-8-sig')
        df['날짜'] = pd.to_datetime(df['날짜'])
        df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

        vix = pd.read_csv(self.vix_csv, parse_dates=[0], encoding='utf-8-sig')
        vix.columns = ['날짜','VIX']
        hys = pd.read_csv(self.hys_csv, parse_dates=[0], encoding='utf-8-sig')
        hys.columns = ['날짜','HYS']

        df = df.merge(vix, on='날짜', how='left').merge(hys, on='날짜', how='left')
        df['거래량_log'] = np.log1p(df['거래량'])
        return df

# ────────────────────────────────────────────────────────────────
# 2) Strategy: 매크로/기술적 매수·매도 로직 캡슐화
# ────────────────────────────────────────────────────────────────
class Strategy:
    def __init__(self, rsi_buy_th, boll_buf, tw_price_th, tm_price_th,
                 tw_vol_th, tm_vol_th, rsi_sell_th):
        self.rsi_buy_th   = rsi_buy_th
        self.boll_buf     = boll_buf
        self.tw_price_th  = tw_price_th
        self.tm_price_th  = tm_price_th
        self.tw_vol_th    = tw_vol_th
        self.tm_vol_th    = tm_vol_th
        self.rsi_sell_th  = rsi_sell_th

    def macro_buy(self, row):
        # VIX >= 60 → 전액 매수 신호
        return (not np.isnan(row['VIX'])) and row['VIX'] >= 60

#     def macro_sell(self, row):
#         # HYS >= 7 → 전량 매도 신호
#         return (not np.isnan(row['HYS'])) and row['HYS'] >= 7

    def tech_buy(self, row):
        # 9개 기술적 조건
        return all([
            row['RSI (14일)']           < self.rsi_buy_th,
            row['종가']                 < row['볼린저밴드 하단'] * (1 + self.boll_buf),
            row['MACD']                 > row['MACD 시그널'],
            row['SMA 5일']              > row['SMA 10일'],
            row['SMA 5일']              < row['SMA 60일'],
            row['가격 상승률 (2주)']     < self.tw_price_th,
            row['가격 상승률 (3개월)']   < self.tm_price_th,
            row['거래량 상승률 (2주)']   < self.tw_vol_th,
            row['거래량 상승률 (3개월)'] < self.tm_vol_th
        ])

    def tech_sell(self, row):
        # RSI > 매도 임계치 → 전량 매도 신호
        return row['RSI (14일)'] > self.rsi_sell_th

# ────────────────────────────────────────────────────────────────
# 3) Backtester: 그리드 ROI 산출 & Heatmap
# ────────────────────────────────────────────────────────────────
class Backtester:
    def __init__(self, df, strategy, cooldown_days=30):
        self.df = df
        self.strat = strategy
        self.cooldown_days = cooldown_days

    def run(self, buy_frac, sell_frac):
        cash = shares = total_injected = 0.0
        cooldown = 0
        for _, row in self.df.iterrows():
            price = row['종가']
            if cooldown>0: cooldown-=1

#             # 1) Macro sell
#             if shares>0 and self.strat.macro_sell(row):
#                 shares_sold = shares * sell_frac
#                 cash += shares_sold * price
#                 shares -= shares_sold
#                 continue

            # 2) Macro buy
            if shares==0 and cooldown==0 and self.strat.macro_buy(row):
                cash += 10_000.0
                total_injected += 10_000.0
                bought = (cash * buy_frac)/price
                cash -= bought*price
                shares += bought
                cooldown = self.cooldown_days
                continue

            # 3) Tech buy
            if shares==0 and cooldown==0 and self.strat.tech_buy(row):
                cash += 10_000.0
                total_injected += 10_000.0
                # bridge sell
                bridge = shares*sell_frac
                cash += bridge*price
                shares -= bridge
                # new buy
                bought = (cash*buy_frac)/price
                cash -= bought*price
                shares += bought
                cooldown = self.cooldown_days
                continue

            # 4) Tech sell
            if shares>0 and self.strat.tech_sell(row):
                sold = shares*sell_frac
                cash += sold*price
                shares -= sold
                continue

        final_val = cash + shares*self.df.iloc[-1]['종가']
        return (final_val - total_injected)/total_injected*100

    def grid_search(self, fracs):
        records = []
        for b in fracs:
            for s in fracs:
                roi = self.run(b, s)
                records.append({'buy_frac':b,'sell_frac':s,'ROI (%)':round(roi,2)})
        return pd.DataFrame(records)

    def plot_heatmap(self, res_df, path):
        heat = res_df.pivot(index='sell_frac', columns='buy_frac', values='ROI (%)')
        plt.figure(figsize=(8,6))
        sns.heatmap(heat, annot=True, fmt=".2f", cmap='viridis',
                    cbar_kws={'label':'ROI (%)'})
        plt.title('Macro+Rule Fraction Heatmap')
        plt.xlabel('Buy Fraction'); plt.ylabel('Sell Fraction')
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()

# ────────────────────────────────────────────────────────────────
# 4) TradeSimulator: 최적 비율로 상세 트레이드 로그
# ────────────────────────────────────────────────────────────────
class TradeSimulator(Backtester):
    def simulate(self, buy_frac, sell_frac):
        trade_log = []
        cash = shares = total_injected = 0.0
        cooldown = 0

        for _, row in self.df.iterrows():
            date, price = row['날짜'], row['종가']
            if cooldown>0: cooldown-=1

#             # macro sell
#             if shares>0 and self.strat.macro_sell(row):
#                 action, trigger = 'SELL_MACRO_HYS', 'HYS>=7'
#                 qty = shares*sell_frac
#                 cash+=qty*price; shares-=qty

            # macro buy
            elif shares==0 and cooldown==0 and self.strat.macro_buy(row):
                action, trigger = 'BUY_MACRO_VIX', 'VIX>=60'
                cash+=10000.0; total_injected+=10000.0
                qty = (cash*buy_frac)/price
                cash-=qty*price; shares+=qty
                cooldown=self.cooldown_days

            # tech buy
            elif shares==0 and cooldown==0 and self.strat.tech_buy(row):
                action, trigger = 'BUY_RULE', 'TECH_RULE'
                cash+=10000.0; total_injected+=10000.0
                bridge = shares*sell_frac
                cash+=bridge*price; shares-=bridge
                qty = (cash*buy_frac)/price
                cash-=qty*price; shares+=qty
                cooldown=self.cooldown_days

            # tech sell
            elif shares>0 and self.strat.tech_sell(row):
                action, trigger = 'SELL_RULE', 'RSI>sell_th'
                qty = shares*sell_frac
                cash+=qty*price; shares-=qty

            else:
                continue

            asset = cash + shares*price
            roi_now = (asset-total_injected)/total_injected*100 if total_injected>0 else np.nan

            trade_log.append({
                'date': date,
                'action': action,
                'trigger': trigger,
                'price': round(price,2),
                'shares': round(qty,6),
                'cash_after': round(cash,2),
                'cash_used': round(qty*price,2),
                'extra_cash': round(10000.0 if 'BUY' in action else 0.0,2),
                'cooldown': cooldown,
                'total_injected': round(total_injected,2),
                'asset_value': round(asset,2),
                'ROI_so_far(%)': round(roi_now,2)
            })

        # final liquidate
        if shares>0:
            date, price = self.df.iloc[-1]['날짜'], self.df.iloc[-1]['종가']
            action, trigger = 'LIQUIDATE', 'END'
            qty = shares
            cash+=qty*price; shares=0
            asset = cash
            roi_now = (asset-total_injected)/total_injected*100

            trade_log.append({
                'date': date,
                'action': action,
                'trigger': trigger,
                'price': round(price,2),
                'shares': round(qty,6),
                'cash_after': round(cash,2),
                'cash_used': round(qty*price,2),
                'extra_cash': 0.0,
                'cooldown': cooldown,
                'total_injected': round(total_injected,2),
                'asset_value': round(asset,2),
                'ROI_so_far(%)': round(roi_now,2)
            })

        log_df = pd.DataFrame(trade_log)
        summary = pd.DataFrame([{
            'buy_frac': buy_frac,
            'sell_frac': sell_frac,
            'total_injected': round(total_injected,2),
            'final_asset': round(cash,2),
            'final_ROI_%': round((cash-total_injected)/total_injected*100,2)
        }])
        return log_df, summary

# ────────────────────────────────────────────────────────────────
# 5) Main
# ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    loader = DataLoader(
        price_csv    = input_csv,
        vix_csv      = os.path.join(macro_folder, "VIX.csv"),
        hys_csv      = os.path.join(macro_folder, "High Yied Spread adjusted.csv")
    )
    df_all = loader.load()

    strat = Strategy(
        rsi_buy_th  = params['rsi_buy_th'],
        boll_buf    = params['boll_buffer'],
        tw_price_th = params['tw_price_th'],
        tm_price_th = params['tm_price_th'],
        tw_vol_th   = params['tw_vol_th'],
        tm_vol_th   = params['tm_vol_th'],
        rsi_sell_th = params['rsi_sell_th']
    )

    bt = Backtester(df_all, strat, cooldown_days=30)
    fracs = [i/10 for i in range(1,11)]
    res_df = bt.grid_search(fracs)
    res_df.to_csv(os.path.join(results_folder,"fraction_heatmap.csv"), index=False)
    bt.plot_heatmap(res_df, os.path.join(results_folder,"fraction_heatmap.png"))

    best = res_df.loc[res_df['ROI (%)'].idxmax()]
    best_b, best_s = best['buy_frac'], best['sell_frac']

    sim = TradeSimulator(df_all, strat, cooldown_days=30)
    log_df, summary_df = sim.simulate(best_b, best_s)
    log_df.to_excel(os.path.join(results_folder,"best_trade_log.xlsx"), index=False)
    summary_df.to_excel(os.path.join(results_folder,"best_trade_summary.xlsx"), index=False)

    print("✅ Heatmap, trade log & summary saved in", results_folder)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.preprocessing import MinMaxScaler
import optuna.visualization as vis

# ────────────────────────────────────────────────────────────────
# 0) 경로 & 파라미터 정의
# ────────────────────────────────────────────────────────────────
input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

param_str = """
rsi_buy_th:   51.1987
boll_buffer:  0.0611
tw_price_th:  0.1096
tm_price_th: 33.3657
tw_vol_th:   83.8030
tm_vol_th:   48.7716
rsi_sell_th: 96.9792
"""
params = {}
for line in param_str.strip().splitlines():
    k, v = line.split(':')
    params[k.strip()] = float(v.strip())

# ────────────────────────────────────────────────────────────────
# 1) DataLoader
# ────────────────────────────────────────────────────────────────
class DataLoader:
    def __init__(self, price_csv, vix_csv, hys_csv):
        self.price_csv = price_csv
        self.vix_csv   = vix_csv
        self.hys_csv   = hys_csv

    def load(self):
        df = pd.read_csv(self.price_csv, encoding='utf-8-sig')
        df['날짜'] = pd.to_datetime(df['날짜'])
        df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

        vix = pd.read_csv(self.vix_csv, parse_dates=[0], encoding='utf-8-sig')
        vix.columns = ['날짜','VIX']
        hys = pd.read_csv(self.hys_csv, parse_dates=[0], encoding='utf-8-sig')
        hys.columns = ['날짜','HYS']

        df = df.merge(vix, on='날짜', how='left')\
               .merge(hys, on='날짜', how='left')
        df['거래량_log'] = np.log1p(df['거래량'])
        return df

# ────────────────────────────────────────────────────────────────
# 2) Strategy
# ────────────────────────────────────────────────────────────────
class Strategy:
    def __init__(self, rsi_buy_th, boll_buf, tw_price_th, tm_price_th,
                 tw_vol_th, tm_vol_th, rsi_sell_th):
        self.rsi_buy_th  = rsi_buy_th
        self.boll_buf    = boll_buf
        self.tw_price_th = tw_price_th
        self.tm_price_th = tm_price_th
        self.tw_vol_th   = tw_vol_th
        self.tm_vol_th   = tm_vol_th
        self.rsi_sell_th = rsi_sell_th

    def macro_buy(self, row):
        return (not np.isnan(row['VIX'])) and row['VIX'] >= 60

    def tech_buy(self, row):
        return all([
            row['RSI (14일)']           < self.rsi_buy_th,
            row['종가']                 < row['볼린저밴드 하단']*(1+self.boll_buf),
            row['MACD']                 > row['MACD 시그널'],
            row['SMA 5일']              > row['SMA 10일'],
            row['SMA 5일']              < row['SMA 60일'],
            row['가격 상승률 (2주)']     < self.tw_price_th,
            row['가격 상승률 (3개월)']   < self.tm_price_th,
            row['거래량 상승률 (2주)']   < self.tw_vol_th,
            row['거래량 상승률 (3개월)'] < self.tm_vol_th
        ])

    def tech_sell(self, row):
        return row['RSI (14일)'] > self.rsi_sell_th

# ────────────────────────────────────────────────────────────────
# 3) Backtester
# ────────────────────────────────────────────────────────────────
class Backtester:
    def __init__(self, df, strat, cooldown_days=30):
        self.df            = df
        self.strat         = strat
        self.cooldown_days = cooldown_days

    def run(self, buy_frac, sell_frac, inject_amt=10000.0):
        cash = 0.0
        shares = 0.0
        total_injected = 0.0
        cooldown = 0

        # if no injection scenario, seed initial cash
        if inject_amt == 0:
            cash = 10_000.0
            total_injected = 10_000.0

        for _, row in self.df.iterrows():
            price = row['종가']
            if cooldown>0:
                cooldown -= 1

            # Macro buy
            if shares==0 and cooldown==0 and self.strat.macro_buy(row):
                if inject_amt>0:
                    cash += inject_amt
                    total_injected += inject_amt
                bought = (cash * buy_frac)/price
                cash  -= bought * price
                shares += bought
                cooldown = self.cooldown_days
                continue

            # Tech buy
            if shares==0 and cooldown==0 and self.strat.tech_buy(row):
                if inject_amt>0:
                    cash += inject_amt
                    total_injected += inject_amt
                # bridge
                bridge = shares * sell_frac
                cash   += bridge * price
                shares -= bridge
                bought = (cash * buy_frac)/price
                cash  -= bought * price
                shares += bought
                cooldown = self.cooldown_days
                continue

            # Tech sell
            if shares>0 and self.strat.tech_sell(row):
                sold = shares * sell_frac
                cash   += sold * price
                shares -= sold
                continue

        final_val = cash + shares * self.df.iloc[-1]['종가']
        return (final_val - total_injected)/total_injected*100

    def grid_search(self, fracs):
        rec = []
        for b in fracs:
            for s in fracs:
                roi = self.run(b, s)
                rec.append({'buy_frac':b, 'sell_frac':s, 'ROI (%)':round(roi,2)})
        return pd.DataFrame(rec)

    def plot_heatmap(self, df, path):
        mat = df.pivot(index='sell_frac', columns='buy_frac', values='ROI (%)')
        plt.figure(figsize=(8,6))
        sns.heatmap(mat, annot=True, fmt=".2f", cmap='viridis',
                    cbar_kws={'label':'ROI (%)'})
        plt.title('Fraction Heatmap')
        plt.xlabel('Buy Fraction'); plt.ylabel('Sell Fraction')
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()

# ────────────────────────────────────────────────────────────────
# 4) TradeSimulator
# ────────────────────────────────────────────────────────────────
class TradeSimulator(Backtester):
    def simulate(self, buy_frac, sell_frac, inject_amt=10000.0):
        log = []
        cash = 0.0
        shares = 0.0
        total_injected = 0.0
        cooldown = 0

        if inject_amt == 0:
            cash = 10_000.0
            total_injected = 10_000.0

        for _, row in self.df.iterrows():
            date, price = row['날짜'], row['종가']
            if cooldown>0:
                cooldown -= 1

            # Macro buy
            if shares==0 and cooldown==0 and self.strat.macro_buy(row):
                action, trigger = 'BUY_MACRO', 'VIX>=60'
                if inject_amt>0:
                    cash += inject_amt
                    total_injected += inject_amt
                qty = (cash * buy_frac)/price
                cash     -= qty*price
                shares   += qty
                cooldown  = self.cooldown_days

            # Tech buy
            elif shares==0 and cooldown==0 and self.strat.tech_buy(row):
                action, trigger = 'BUY_RULE', 'TECH_RULE'
                if inject_amt>0:
                    cash += inject_amt
                    total_injected += inject_amt
                bridge = shares*sell_frac
                cash   += bridge*price
                shares -= bridge
                qty     = (cash * buy_frac)/price
                cash   -= qty*price
                shares += qty
                cooldown = self.cooldown_days

            # Tech sell
            elif shares>0 and self.strat.tech_sell(row):
                action, trigger = 'SELL_RULE', 'RSI>sell_th'
                qty = shares*sell_frac
                cash   += qty*price
                shares -= qty

            else:
                continue

            asset = cash + shares*price
            roi_now = (asset-total_injected)/total_injected*100 if total_injected>0 else np.nan

            log.append({
                'date':           date,
                'action':         action,
                'trigger':        trigger,
                'price':          round(price,2),
                'shares':         round(qty,6),
                'cash_after':     round(cash,2),
                'cash_used':      round(qty*price,2),
                'extra_cash':     round(inject_amt if 'BUY' in action else 0,2),
                'cooldown':       cooldown,
                'total_injected': round(total_injected,2),
                'asset_value':    round(asset,2),
                'ROI_so_far(%)':  round(roi_now,2)
            })

        # final liquidate
        if shares>0:
            date, price = self.df.iloc[-1]['날짜'], self.df.iloc[-1]['종가']
            action, trigger = 'LIQUIDATE','END'
            qty = shares
            cash += qty*price
            shares = 0.0
            asset = cash
            roi_now = (asset-total_injected)/total_injected*100

            log.append({
                'date':           date,
                'action':         action,
                'trigger':        trigger,
                'price':          round(price,2),
                'shares':         round(qty,6),
                'cash_after':     round(cash,2),
                'cash_used':      round(qty*price,2),
                'extra_cash':     0.0,
                'cooldown':       cooldown,
                'total_injected': round(total_injected,2),
                'asset_value':    round(asset,2),
                'ROI_so_far(%)':  round(roi_now,2)
            })

        return pd.DataFrame(log), pd.DataFrame([{
            'buy_frac':       buy_frac,
            'sell_frac':      sell_frac,
            'total_injected': round(total_injected,2),
            'final_asset':    round(cash,2),
            'final_ROI_%':    round((cash-total_injected)/total_injected*100,2)
        }])

# ────────────────────────────────────────────────────────────────
# 5) Main
# ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    loader = DataLoader(
        price_csv = input_csv,
        vix_csv   = os.path.join(macro_folder, "VIX.csv"),
        hys_csv   = os.path.join(macro_folder, "High Yied Spread adjusted.csv")
    )
    df_all = loader.load()

    strat = Strategy(
        rsi_buy_th  = params['rsi_buy_th'],
        boll_buf    = params['boll_buffer'],
        tw_price_th = params['tw_price_th'],
        tm_price_th = params['tm_price_th'],
        tw_vol_th   = params['tw_vol_th'],
        tm_vol_th   = params['tm_vol_th'],
        rsi_sell_th = params['rsi_sell_th']
    )

    bt = Backtester(df_all, strat, cooldown_days=30)
    fracs = [i/10 for i in range(1,11)]
    res_df = bt.grid_search(fracs)
    res_df.to_csv(os.path.join(results_folder,"fraction_heatmap.csv"), index=False)
    bt.plot_heatmap(res_df, os.path.join(results_folder,"fraction_heatmap.png"))

    best = res_df.loc[res_df['ROI (%)'].idxmax()]
    best_b, best_s = best['buy_frac'], best['sell_frac']

    sim = TradeSimulator(df_all, strat, cooldown_days=30)
    # 1) With extra-cash injection
    log_inj, sum_inj = sim.simulate(best_b, best_s, inject_amt=10000.0)
    log_inj.to_excel(os.path.join(results_folder,"best_trade_log_inject.xlsx"), index=False)
    sum_inj.to_excel(os.path.join(results_folder,"best_trade_summary_inject.xlsx"), index=False)

    # 2) Only initial $10k, no extra injection
    log_noinj, sum_noinj = sim.simulate(best_b, best_s, inject_amt=0.0)
    log_noinj.to_excel(os.path.join(results_folder,"best_trade_log_no_inject.xlsx"), index=False)
    sum_noinj.to_excel(os.path.join(results_folder,"best_trade_summary_no_inject.xlsx"), index=False)

    print("✅ Heatmap, logs & summaries saved in", results_folder)


In [ ]:
# 모든거 업데이트

In [ ]:
import os
import pandas as pd
import numpy as np
import optuna

# ─── 1) 경로 설정 ─────────────────────────────
input_csv = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ─── 2) 데이터 불러오기 ───────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
vix_df.columns = ['날짜', 'VIX']
hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"), parse_dates=[0], encoding='utf-8-sig')
hys_df.columns = ['날짜', 'HYS']

df = df.merge(vix_df, on='날짜', how='left')
df = df.merge(hys_df, on='날짜', how='left')
df['거래량_log'] = np.log1p(df['거래량'])

# ─── 3) 조건 클래스 정의 ───────────────────────
class StrategyCondition:
    def check(self, row): raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th

class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row): return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct

class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# ─── 4) 백테스트 함수 ─────────────────────────
def backtest_with_tracking(rsi_buy_th, boll_buffer, tw_price_th, tm_price_th, tw_vol_th, tm_vol_th, rsi_sell_th):
    cash, shares = 10000.0, 0.0
    total_invested = 10000.0
    buy_conds = [
        RSIBelow(rsi_buy_th),
        BollingerNearLower(boll_buffer),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(tw_price_th),
        ThreeMonthPriceLow(tm_price_th),
        TwoWeekVolumeLow(tw_vol_th),
        ThreeMonthVolumeLow(tm_vol_th),
    ]
    sell_conds = [RSISell(rsi_sell_th)]
    records = []

    for _, row in df.iterrows():
        price, date, vix = row['종가'], row['날짜'], row['VIX']
        buy_signal = all(cond.check(row) for cond in buy_conds) and (not pd.isna(vix) and vix >= 60)
        sell_signal = any(cond.check(row) for cond in sell_conds)

        if buy_signal:
            cash += 10000
            total_invested += 10000
            buy_shares = 10000 / price
            shares += buy_shares
            cash -= 10000
            records.append([date, "BUY", price, buy_shares, cash, shares, (cash + shares * price) / total_invested * 100])
        elif sell_signal and shares > 0:
            cash += shares * price
            records.append([date, "SELL", price, shares, cash, 0.0, (cash) / total_invested * 100])
            shares = 0.0

    final_val = cash + shares * df.iloc[-1]['종가']
    roi = (final_val / total_invested) * 100
    return roi, pd.DataFrame(records, columns=["날짜", "액션", "가격", "수량", "현금", "보유주", "ROI(%)"])

# ─── 5) Optuna 최적화 ──────────────────────────
def objective(trial):
    roi, _ = backtest_with_tracking(
        trial.suggest_float("rsi_buy_th",  0, 100),
        trial.suggest_float("boll_buffer", 0, 0.1),
        trial.suggest_float("tw_price_th", 0, 20),
        trial.suggest_float("tm_price_th", 0, 50),
        trial.suggest_float("tw_vol_th",   0, 100),
        trial.suggest_float("tm_vol_th",   0, 300),
        trial.suggest_float("rsi_sell_th", 0, 100),
    )
    return roi

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# ─── 6) 최적 결과로 시뮬레이션 ────────────────
best_params = study.best_params
best_roi, tracking_df = backtest_with_tracking(**best_params)

# ─── 7) 결과 저장 및 출력 ─────────────────────
out_path = os.path.join(results_folder, "📈Best_Trade_Record.xlsx")
tracking_df.to_excel(out_path, index=False)

print("\n✅ Best Parameters:")
for k, v in best_params.items():
    print(f"   {k}: {v:.4f}")
print(f"\n💰 Best ROI: {best_roi:.2f}%")
print(f"📁 Excel 저장 경로: {out_path}")


In [ ]:
⭐️ Best ROI (%): 269501.09
⭐️ Best Params:
   rsi_buy_th: 47.8733
   boll_buffer: 0.0150
   tw_price_th: 10.2913
   tm_price_th: 23.1772
   tw_vol_th: 70.1934
   tm_vol_th: 225.2615
   rsi_sell_th: 96.8699
▶ Validation ROI with best params: 269501.09%

In [ ]:
import pandas as pd
import numpy as np
import os

# ── 1. 파라미터 입력 ──
params = {
    'rsi_buy_th': 49,
    'boll_buffer': 0.0954,
    'tw_price_th': 4.6432,
    'tm_price_th': 49.6071,
    'tw_vol_th': 67.3719,
    'tm_vol_th': 228.6710,
    'rsi_sell_th': 10.5987
}

# ── 2. 저장 경로 설정 ──
save_dir = r"C:\Users\LabPC\OneDrive\주식\Results"
os.makedirs(save_dir, exist_ok=True)

# ── 3. 데이터 로딩 (파일 경로 수정 요망) ──
df = pd.read_csv(r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv", encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')]

vix = pd.read_csv(r"C:\Users\LabPC\OneDrive\주식\Macro Data\VIX.csv", parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜', 'VIX']
df = df.merge(vix, on='날짜', how='left')

# ── 4. 조건 함수 정의 ──
def buy_signal(row):
    return (
        row['RSI (14일)'] < params['rsi_buy_th'] and
        row['종가'] < row['볼린저밴드 하단'] * (1 + params['boll_buffer']) and
        row['MACD'] > row['MACD 시그널'] and
        row['SMA 5일'] > row['SMA 10일'] and
        row['SMA 5일'] < row['SMA 60일'] and
        row['가격 상승률 (2주)'] < params['tw_price_th'] and
        row['가격 상승률 (3개월)'] < params['tm_price_th'] and
        row['거래량 상승률 (2주)'] < params['tw_vol_th'] and
        row['거래량 상승률 (3개월)'] < params['tm_vol_th'] and
        row['VIX'] >= 60
    )

def sell_signal(row):
    return row['RSI (14일)'] > params['rsi_sell_th']

# ── 5. 백테스트 함수 ──
def run_backtest(extra_on_buy=False):
    cash, shares, total_invested = 10000, 0, 10000
    logs = []

    for _, row in df.iterrows():
        price, date = row['종가'], row['날짜']
        if buy_signal(row):
            invest = 10000 if extra_on_buy else (cash if shares == 0 else 0)
            if invest > 0:
                cash += invest if extra_on_buy else 0
                total_invested += invest if extra_on_buy else 0
                shares += invest / price
                cash -= invest
                logs.append([date, "BUY", price, shares, cash, shares * price + cash, (shares * price + cash) / total_invested * 100])
        elif sell_signal(row) and shares > 0:
            cash += shares * price
            logs.append([date, "SELL", price, shares, cash, cash, cash / total_invested * 100])
            shares = 0

    final_value = cash + shares * df.iloc[-1]['종가']
    roi = final_value / total_invested * 100
    return pd.DataFrame(logs, columns=["날짜", "액션", "가격", "보유주", "현금", "총자산", "ROI(%)"]), roi

# ── 6. 실행 및 CSV 저장 ──
df1, roi1 = run_backtest(extra_on_buy=False)
df2, roi2 = run_backtest(extra_on_buy=True)

df1.to_csv(os.path.join(save_dir, "one_time_investment.csv"), index=False, encoding='utf-8-sig')
df2.to_csv(os.path.join(save_dir, "extra_10000_every_buy.csv"), index=False, encoding='utf-8-sig')

print(f"✅ 1회 투자 ROI: {roi1:.2f}% → 저장 위치: {save_dir}\\one_time_investment.csv")
print(f"✅ 매수마다 1만 추가 투자 ROI: {roi2:.2f}% → 저장 위치: {save_dir}\\extra_10000_every_buy.csv")
